# Drug Exposure Diagnostics with OMOPy

This notebook demonstrates `omopy.drug_diagnostics` for running quality checks on drug exposure data, summarising results into the standard `SummarisedResult` format, and producing tables and plots.

**Covered:**
- Available diagnostic checks (`AVAILABLE_CHECKS`)
- Running checks with `execute_checks` (all checks, subset, ingredient filter)
- Examining the `DiagnosticsResult` object
- Summarising with `summarise_drug_diagnostics`
- Table output with `table_drug_diagnostics` (GT and polars formats)
- Plotting with `plot_drug_diagnostics`
- Mock data via `mock_drug_exposure` for richer demonstrations
- Benchmarking with `benchmark_drug_diagnostics`

**Database:** Synthea DuckDB with 27 persons, 663 drug_exposure records, 32 unique drug concepts, empty `drug_strength` table.
Mock data is used for larger-sample demonstrations.

## 1. Configuration

In [1]:
DB_PATH = "../data/synthea.duckdb"
CDM_SCHEMA = "base"
WRITE_SCHEMA = "main"

## 2. Setup

Connect to the CDM and import the drug diagnostics module.

In [2]:
from omopy.connector import cdm_from_con
from omopy.drug_diagnostics import (
    AVAILABLE_CHECKS,
    DiagnosticsResult,
    execute_checks,
    summarise_drug_diagnostics,
    table_drug_diagnostics,
    plot_drug_diagnostics,
    mock_drug_exposure,
    benchmark_drug_diagnostics,
)

print("Imports OK")

Imports OK


In [3]:
cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA, write_schema=WRITE_SCHEMA)
print(cdm)
print(f"Persons: {cdm['person'].count()}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Persons: 27


In [4]:
# Quick look at drug_exposure
de = cdm["drug_exposure"].collect()
print(f"Drug exposure records: {len(de)}")
print(f"Unique drug_concept_ids: {de['drug_concept_id'].n_unique()}")
de.head(5)

Drug exposure records: 663
Unique drug_concept_ids: 32


drug_exposure_id,person_id,drug_concept_id,drug_exposure_start_date,drug_exposure_start_datetime,drug_exposure_end_date,drug_exposure_end_datetime,verbatim_end_date,drug_type_concept_id,stop_reason,refills,quantity,days_supply,sig,route_concept_id,lot_number,provider_id,visit_occurrence_id,visit_detail_id,drug_source_value,drug_source_concept_id,route_source_value,dose_unit_source_value
i64,i64,i32,date,datetime[μs],date,datetime[μs],date,i32,str,i32,i32,i64,str,i32,str,i64,i64,i64,str,i32,str,str
1,1,19125062,2014-08-02,2014-08-02 14:15:57,2014-08-02,2014-08-02 14:15:57,null,32838,null,null,null,null,null,0,"""0""",5,2,1000002,"""665078""",19125062,null,null
2,1,19019979,2023-04-22,2023-04-22 17:20:30,2023-05-10,2023-05-10 17:20:30,2023-05-10,32838,null,null,null,18,null,0,"""0""",5,4,1000004,"""198405""",19019979,null,null
3,1,1594382,2014-08-02,2014-08-02 14:15:57,2014-08-02,2014-08-02 14:15:57,null,32838,null,null,null,null,null,0,"""0""",5,2,1000002,"""1870230""",1594382,null,null
4,2,40160973,2020-10-07,2020-10-07 06:50:57,2020-10-07,2020-10-07 06:50:57,null,32838,null,null,null,null,null,0,"""0""",21,5,1000005,"""854235""",40160973,null,null
5,2,40160997,2020-10-14,2020-10-14 06:50:57,2020-10-14,2020-10-14 06:50:57,null,32838,null,null,null,null,null,0,"""0""",21,5,1000005,"""854252""",40160997,null,null


## 3. Available Checks

`AVAILABLE_CHECKS` lists all 12 diagnostic check names supported by `execute_checks`.

In [5]:
print(f"Number of checks: {len(AVAILABLE_CHECKS)}")
for i, check in enumerate(AVAILABLE_CHECKS, 1):
    print(f"  {i:2d}. {check}")

Number of checks: 12
   1. missing
   2. exposure_duration
   3. type
   4. route
   5. source_concept
   6. days_supply
   7. verbatim_end_date
   8. dose
   9. sig
  10. quantity
  11. days_between
  12. diagnostics_summary


## 4. Executing Diagnostics

`execute_checks` runs diagnostics for one or more ingredient concept IDs.
It resolves descendant drug concepts, fetches records, and runs each enabled check.

We use ingredient concept IDs from the Synthea data: 1367500 (Amoxicillin) and 19133768 (Ibuprofen).
We set `min_cell_count=0` since the Synthea dataset is small.

### 4a. All checks for a single ingredient

In [6]:
result_all = execute_checks(
    cdm,
    ingredient_concept_ids=1367500,  # Amoxicillin
    min_cell_count=0,
)
print(result_all)

results={'missing': shape: (0, 8)
┌─────────────┬────────────┬──────────┬───────────┬──────────┬───────────┬────────────┬────────────┐
│ ingredient_ ┆ ingredient ┆ variable ┆ n_records ┆ n_sample ┆ n_missing ┆ n_not_miss ┆ proportion │
│ concept_id  ┆ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ing        ┆ _missing   │
│ ---         ┆ str        ┆ str      ┆ i64       ┆ i64      ┆ i64       ┆ ---        ┆ ---        │
│ i64         ┆            ┆          ┆           ┆          ┆           ┆ i64        ┆ f64        │
╞═════════════╪════════════╪══════════╪═══════════╪══════════╪═══════════╪════════════╪════════════╡
└─────────────┴────────────┴──────────┴───────────┴──────────┴───────────┴────────────┴────────────┘, 'exposure_duration': shape: (0, 19)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ingredien ┆ ingredien ┆ n_records ┆ n_sample ┆ … ┆ duration_ ┆ duration_ ┆ duration_ ┆ duration_ │
│ t_concept ┆ t     

### 4b. Subset of checks

In [7]:
result_subset = execute_checks(
    cdm,
    ingredient_concept_ids=1367500,
    checks=["missing", "exposure_duration", "type", "route"],
    min_cell_count=0,
)
print(result_subset)
print(f"Checks performed: {result_subset.checks_performed}")

results={'missing': shape: (0, 8)
┌─────────────┬────────────┬──────────┬───────────┬──────────┬───────────┬────────────┬────────────┐
│ ingredient_ ┆ ingredient ┆ variable ┆ n_records ┆ n_sample ┆ n_missing ┆ n_not_miss ┆ proportion │
│ concept_id  ┆ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ing        ┆ _missing   │
│ ---         ┆ str        ┆ str      ┆ i64       ┆ i64      ┆ i64       ┆ ---        ┆ ---        │
│ i64         ┆            ┆          ┆           ┆          ┆           ┆ i64        ┆ f64        │
╞═════════════╪════════════╪══════════╪═══════════╪══════════╪═══════════╪════════════╪════════════╡
└─────────────┴────────────┴──────────┴───────────┴──────────┴───────────┴────────────┴────────────┘, 'exposure_duration': shape: (0, 19)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ingredien ┆ ingredien ┆ n_records ┆ n_sample ┆ … ┆ duration_ ┆ duration_ ┆ duration_ ┆ duration_ │
│ t_concept ┆ t     

### 4c. Multiple ingredients

In [8]:
result_multi = execute_checks(
    cdm,
    ingredient_concept_ids=[1367500, 19133768],  # Amoxicillin + Ibuprofen
    checks=["missing", "exposure_duration", "type", "diagnostics_summary"],
    min_cell_count=0,
)
print(result_multi)
print(f"Ingredients: {result_multi.ingredient_concepts}")

results={'missing': shape: (15, 8)
┌────────────┬────────────┬────────────┬───────────┬──────────┬───────────┬────────────┬───────────┐
│ ingredient ┆ ingredient ┆ variable   ┆ n_records ┆ n_sample ┆ n_missing ┆ n_not_miss ┆ proportio │
│ _concept_i ┆ ---        ┆ ---        ┆ ---       ┆ ---      ┆ ---       ┆ ing        ┆ n_missing │
│ d          ┆ str        ┆ str        ┆ i64       ┆ i64      ┆ i64       ┆ ---        ┆ ---       │
│ ---        ┆            ┆            ┆           ┆          ┆           ┆ i64        ┆ f64       │
│ i64        ┆            ┆            ┆           ┆          ┆           ┆            ┆           │
╞════════════╪════════════╪════════════╪═══════════╪══════════╪═══════════╪════════════╪═══════════╡
│ 19133768   ┆ acetaminop ┆ drug_expos ┆ 1         ┆ 1        ┆ 0         ┆ 1          ┆ 0.0       │
│            ┆ hen 750 MG ┆ ure_id     ┆           ┆          ┆           ┆            ┆           │
│            ┆ / hydroco… ┆            ┆           ┆    

## 5. Examining DiagnosticsResult

`DiagnosticsResult` holds a dict of Polars DataFrames (one per check) plus execution metadata.
Access individual check results with dict-like indexing.

In [9]:
# Metadata
print(f"CDM name:        {result_all.cdm_name}")
print(f"Checks run:      {result_all.checks_performed}")
print(f"Ingredients:     {result_all.ingredient_concepts}")
print(f"Sample size:     {result_all.sample_size}")
print(f"Min cell count:  {result_all.min_cell_count}")
print(f"Execution time:  {result_all.execution_time_seconds:.2f}s")

CDM name:        dbt-synthea
Checks run:      ('missing', 'exposure_duration', 'type', 'route', 'source_concept', 'days_supply', 'verbatim_end_date', 'dose', 'sig', 'quantity', 'days_between', 'diagnostics_summary')
Ingredients:     {1367500: 'Unknown (1367500)'}
Sample size:     10000
Min cell count:  0
Execution time:  0.05s


In [10]:
# List available check results
for name in result_all.keys():
    df = result_all[name]
    print(f"  {name:25s} -> {df.height} rows, {df.width} cols")

  missing                   -> 0 rows, 8 cols
  exposure_duration         -> 0 rows, 19 cols
  type                      -> 0 rows, 8 cols
  route                     -> 0 rows, 8 cols
  source_concept            -> 0 rows, 9 cols
  days_supply               -> 0 rows, 20 cols
  verbatim_end_date         -> 0 rows, 10 cols
  dose                      -> 1 rows, 7 cols
  sig                       -> 0 rows, 7 cols
  quantity                  -> 0 rows, 17 cols
  days_between              -> 0 rows, 19 cols
  diagnostics_summary       -> 1 rows, 12 cols


In [11]:
# Missing values check
print("Missing values check:")
print(result_all["missing"])

Missing values check:
shape: (0, 8)
┌─────────────┬────────────┬──────────┬───────────┬──────────┬───────────┬────────────┬────────────┐
│ ingredient_ ┆ ingredient ┆ variable ┆ n_records ┆ n_sample ┆ n_missing ┆ n_not_miss ┆ proportion │
│ concept_id  ┆ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ing        ┆ _missing   │
│ ---         ┆ str        ┆ str      ┆ i64       ┆ i64      ┆ i64       ┆ ---        ┆ ---        │
│ i64         ┆            ┆          ┆           ┆          ┆           ┆ i64        ┆ f64        │
╞═════════════╪════════════╪══════════╪═══════════╪══════════╪═══════════╪════════════╪════════════╡
└─────────────┴────────────┴──────────┴───────────┴──────────┴───────────┴────────────┴────────────┘


In [12]:
# Exposure duration check
print("Exposure duration check:")
print(result_all["exposure_duration"])

Exposure duration check:
shape: (0, 19)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ingredien ┆ ingredien ┆ n_records ┆ n_sample ┆ … ┆ duration_ ┆ duration_ ┆ duration_ ┆ duration_ │
│ t_concept ┆ t         ┆ ---       ┆ ---      ┆   ┆ min       ┆ max       ┆ count     ┆ count_mis │
│ _id       ┆ ---       ┆ i64       ┆ i64      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ sing      │
│ ---       ┆ str       ┆           ┆          ┆   ┆ f64       ┆ f64       ┆ i64       ┆ ---       │
│ i64       ┆           ┆           ┆          ┆   ┆           ┆           ┆           ┆ i64       │
╞═══════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
└───────────┴───────────┴───────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [13]:
# Drug type frequency check
print("Drug type check:")
print(result_all["type"])

Drug type check:
shape: (0, 8)
┌──────────────┬────────────┬──────────────┬───────────┬───────────┬──────────┬───────┬────────────┐
│ ingredient_c ┆ ingredient ┆ drug_type_co ┆ drug_type ┆ n_records ┆ n_sample ┆ count ┆ proportion │
│ oncept_id    ┆ ---        ┆ ncept_id     ┆ ---       ┆ ---       ┆ ---      ┆ ---   ┆ ---        │
│ ---          ┆ str        ┆ ---          ┆ str       ┆ i64       ┆ i64      ┆ i64   ┆ f64        │
│ i64          ┆            ┆ i64          ┆           ┆           ┆          ┆       ┆            │
╞══════════════╪════════════╪══════════════╪═══════════╪═══════════╪══════════╪═══════╪════════════╡
└──────────────┴────────────┴──────────────┴───────────┴───────────┴──────────┴───────┴────────────┘


In [14]:
# Diagnostics summary (aggregated across checks)
print("Diagnostics summary:")
print(result_all["diagnostics_summary"])

Diagnostics summary:
shape: (1, 12)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ingredien ┆ ingredien ┆ n_records ┆ n_sample ┆ … ┆ median_da ┆ median_qu ┆ proportio ┆ proportio │
│ t_concept ┆ t         ┆ ---       ┆ ---      ┆   ┆ ys_supply ┆ antity    ┆ n_with_do ┆ n_verbati │
│ _id       ┆ ---       ┆ i64       ┆ i64      ┆   ┆ ---       ┆ ---       ┆ se        ┆ m_end_dat │
│ ---       ┆ str       ┆           ┆          ┆   ┆ null      ┆ null      ┆ ---       ┆ e_m…      │
│ i64       ┆           ┆           ┆          ┆   ┆           ┆           ┆ f64       ┆ ---       │
│           ┆           ┆           ┆          ┆   ┆           ┆           ┆           ┆ null      │
╞═══════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1367500   ┆ Unknown   ┆ 0         ┆ 0        ┆ … ┆ null      ┆ null      ┆ 0.0       ┆ null      │
│           ┆ (1367500) ┆           ┆          ┆   ┆   

## 6. Summarise Diagnostics

`summarise_drug_diagnostics` converts the `DiagnosticsResult` into the standard 13-column `SummarisedResult` format used across OMOPy.

In [15]:
sr = summarise_drug_diagnostics(result_all)
print(type(sr))
print(f"Rows: {sr.data.height}")
print(f"Settings: {sr.settings.height} rows")

<class 'omopy.generics.summarised_result.SummarisedResult'>
Rows: 14
Settings: 12 rows


In [16]:
# View the settings (one row per check)
print(sr.settings)

shape: (12, 6)
┌───────────┬──────────────────┬──────────────────┬─────────────────┬─────────────┬────────────────┐
│ result_id ┆ result_type      ┆ package_name     ┆ package_version ┆ sample_size ┆ min_cell_count │
│ ---       ┆ ---              ┆ ---              ┆ ---             ┆ ---         ┆ ---            │
│ i64       ┆ str              ┆ str              ┆ str             ┆ str         ┆ str            │
╞═══════════╪══════════════════╪══════════════════╪═════════════════╪═════════════╪════════════════╡
│ 1         ┆ drug_diagnostics ┆ omopy.drug_diagn ┆ 0.1.0           ┆ 10000       ┆ 0              │
│           ┆ _missing         ┆ ostics           ┆                 ┆             ┆                │
│ 2         ┆ drug_diagnostics ┆ omopy.drug_diagn ┆ 0.1.0           ┆ 10000       ┆ 0              │
│           ┆ _exposure_dura…  ┆ ostics           ┆                 ┆             ┆                │
│ 3         ┆ drug_diagnostics ┆ omopy.drug_diagn ┆ 0.1.0           ┆ 10000 

In [17]:
# View the first few rows of the summarised data
print(sr.data.head(15))

shape: (14, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ group_nam ┆ group_lev ┆ … ┆ estimate_ ┆ estimate_ ┆ additiona ┆ addition │
│ ---       ┆ ---       ┆ e         ┆ el        ┆   ┆ type      ┆ value     ┆ l_name    ┆ al_level │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 8         ┆ dbt-synth ┆ ingredien ┆ 1367500   ┆ … ┆ integer   ┆ 0         ┆ overall   ┆ overall  │
│           ┆ ea        ┆ t_concept ┆ &&&       ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆ _id &&&   ┆ Unknown   ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆ ingr…     ┆ (1367500) ┆   ┆           ┆          

## 7. Diagnostics Table

`table_drug_diagnostics` formats the `SummarisedResult` as a display-ready table.
Use `type="polars"` for a DataFrame or `type="gt"` for a `great_tables.GT` object.

In [18]:
# Polars table for the "missing" check
tbl_missing = table_drug_diagnostics(sr, check="missing", type="polars")
print("Missing values table (polars):")
print(tbl_missing)

Missing values table (polars):
shape: (0, 5)
┌──────────┬───────────────┬────────────────┬───────────────┬────────────────┐
│ cdm_name ┆ variable_name ┆ variable_level ┆ estimate_name ┆ estimate_value │
│ ---      ┆ ---           ┆ ---            ┆ ---           ┆ ---            │
│ str      ┆ str           ┆ str            ┆ str           ┆ str            │
╞══════════╪═══════════════╪════════════════╪═══════════════╪════════════════╡
└──────────┴───────────────┴────────────────┴───────────────┴────────────────┘


In [19]:
# Polars table for "exposure_duration"
tbl_duration = table_drug_diagnostics(sr, check="exposure_duration", type="polars")
print("Exposure duration table (polars):")
print(tbl_duration)

Exposure duration table (polars):
shape: (0, 5)
┌──────────┬───────────────┬────────────────┬───────────────┬────────────────┐
│ cdm_name ┆ variable_name ┆ variable_level ┆ estimate_name ┆ estimate_value │
│ ---      ┆ ---           ┆ ---            ┆ ---           ┆ ---            │
│ str      ┆ str           ┆ str            ┆ str           ┆ str            │
╞══════════╪═══════════════╪════════════════╪═══════════════╪════════════════╡
└──────────┴───────────────┴────────────────┴───────────────┴────────────────┘


In [20]:
# GT table for the "type" check (renders as rich HTML)
tbl_type_gt = table_drug_diagnostics(sr, check="type", type="gt")
tbl_type_gt

cdm_name,variable_level,estimate_name,estimate_value


## 8. Diagnostics Plot

`plot_drug_diagnostics` creates interactive Plotly figures. It supports:
- Bar charts for categorical checks (`missing`, `type`, `route`, `sig`, `source_concept`)
- Box plots for quantile checks (`exposure_duration`, `days_supply`, `quantity`, `days_between`)

In [21]:
# Missing values bar chart
fig_missing = plot_drug_diagnostics(sr, check="missing")
fig_missing.show()

In [22]:
# Exposure duration box plot
fig_duration = plot_drug_diagnostics(sr, check="exposure_duration")
fig_duration.show()

In [23]:
# Drug type frequency bar chart
fig_type = plot_drug_diagnostics(sr, check="type")
fig_type.show()

## 9. Using Mock Data

The Synthea dataset is small (27 persons) and has empty `drug_strength` / NULL `sig` fields.
`mock_drug_exposure` generates a `DiagnosticsResult` with realistic synthetic data, useful for demonstrating the full range of checks and visualisations.

In [24]:
mock_result = mock_drug_exposure(
    n_ingredients=3,
    n_records_per_ingredient=500,
    seed=42,
)
print(mock_result)

results={'missing': shape: (45, 8)
┌────────────┬────────────┬────────────┬───────────┬──────────┬───────────┬────────────┬───────────┐
│ ingredient ┆ ingredient ┆ variable   ┆ n_records ┆ n_sample ┆ n_missing ┆ n_not_miss ┆ proportio │
│ _concept_i ┆ ---        ┆ ---        ┆ ---       ┆ ---      ┆ ---       ┆ ing        ┆ n_missing │
│ d          ┆ str        ┆ str        ┆ i64       ┆ i64      ┆ i64       ┆ ---        ┆ ---       │
│ ---        ┆            ┆            ┆           ┆          ┆           ┆ i64        ┆ f64       │
│ i64        ┆            ┆            ┆           ┆          ┆           ┆            ┆           │
╞════════════╪════════════╪════════════╪═══════════╪══════════╪═══════════╪════════════╪═══════════╡
│ 1125315    ┆ Acetaminop ┆ drug_expos ┆ 500       ┆ 500      ┆ 12        ┆ 488        ┆ 0.024     │
│            ┆ hen        ┆ ure_id     ┆           ┆          ┆           ┆            ┆           │
│ 1125315    ┆ Acetaminop ┆ person_id  ┆ 500       ┆ 500

In [25]:
# List mock check results
for name in mock_result.keys():
    df = mock_result[name]
    print(f"  {name:25s} -> {df.height} rows")

  missing                   -> 45 rows
  exposure_duration         -> 3 rows
  type                      -> 7 rows
  route                     -> 7 rows
  source_concept            -> 11 rows
  days_supply               -> 3 rows
  verbatim_end_date         -> 3 rows
  dose                      -> 3 rows
  sig                       -> 7 rows
  quantity                  -> 3 rows
  days_between              -> 3 rows
  diagnostics_summary       -> 3 rows


In [26]:
# Summarise mock results
mock_sr = summarise_drug_diagnostics(mock_result)
print(f"Mock summarised rows: {mock_sr.data.height}")
print(mock_sr.settings)

Mock summarised rows: 481
shape: (12, 6)
┌───────────┬──────────────────┬──────────────────┬─────────────────┬─────────────┬────────────────┐
│ result_id ┆ result_type      ┆ package_name     ┆ package_version ┆ sample_size ┆ min_cell_count │
│ ---       ┆ ---              ┆ ---              ┆ ---             ┆ ---         ┆ ---            │
│ i64       ┆ str              ┆ str              ┆ str             ┆ str         ┆ str            │
╞═══════════╪══════════════════╪══════════════════╪═════════════════╪═════════════╪════════════════╡
│ 1         ┆ drug_diagnostics ┆ omopy.drug_diagn ┆ 0.1.0           ┆ 500         ┆ 5              │
│           ┆ _missing         ┆ ostics           ┆                 ┆             ┆                │
│ 2         ┆ drug_diagnostics ┆ omopy.drug_diagn ┆ 0.1.0           ┆ 500         ┆ 5              │
│           ┆ _exposure_dura…  ┆ ostics           ┆                 ┆             ┆                │
│ 3         ┆ drug_diagnostics ┆ omopy.drug_diagn 

In [27]:
# Table from mock data
mock_tbl = table_drug_diagnostics(mock_sr, check="missing", type="polars")
print("Mock missing values table:")
print(mock_tbl)

Mock missing values table:
shape: (180, 6)
┌───────────────┬────────────────┬────────────────┬────────────────┬───────────────┬───────────────┐
│ variable_name ┆ variable_level ┆ estimate_name  ┆ [header_name]c ┆ ingredient_co ┆ ingredient    │
│ ---           ┆ ---            ┆ ---            ┆ dm_name        ┆ ncept_id      ┆ ---           │
│ str           ┆ str            ┆ str            ┆ [header_…      ┆ ---           ┆ str           │
│               ┆                ┆                ┆ ---            ┆ str           ┆               │
│               ┆                ┆                ┆ str            ┆               ┆               │
╞═══════════════╪════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╡
│ missing       ┆ drug_exposure_ ┆ n_records      ┆ 500            ┆ 1125315       ┆ Acetaminophen │
│               ┆ id             ┆                ┆                ┆               ┆               │
│ missing       ┆ drug_exposure_ ┆ n_missing    

In [28]:
# Plot mock missing values (3 ingredients side by side)
fig_mock_missing = plot_drug_diagnostics(mock_sr, check="missing")
fig_mock_missing.show()

In [29]:
# Plot mock exposure duration (box plots per ingredient)
fig_mock_dur = plot_drug_diagnostics(mock_sr, check="exposure_duration")
fig_mock_dur.show()

In [30]:
# Plot mock route frequencies
fig_mock_route = plot_drug_diagnostics(mock_sr, check="route")
fig_mock_route.show()

In [31]:
# Plot mock quantity distribution
fig_mock_qty = plot_drug_diagnostics(mock_sr, check="quantity")
fig_mock_qty.show()

In [32]:
# GT table from mock data — diagnostics summary
mock_tbl_summary = table_drug_diagnostics(mock_sr, check="diagnostics_summary", type="gt")
mock_tbl_summary

GT(_tbl_data=          variable_name                        variable_level  \
0   diagnostics_summary                             n_records   
1   diagnostics_summary                              n_sample   
2   diagnostics_summary                             n_persons   
3   diagnostics_summary               mean_proportion_missing   
4   diagnostics_summary                  median_duration_days   
5   diagnostics_summary                   n_negative_duration   
6   diagnostics_summary                    median_days_supply   
7   diagnostics_summary                       median_quantity   
8   diagnostics_summary                  proportion_with_dose   
9   diagnostics_summary  proportion_verbatim_end_date_missing   
10  diagnostics_summary                             n_records   
11  diagnostics_summary                              n_sample   
12  diagnostics_summary                             n_persons   
13  diagnostics_summary               mean_proportion_missing   
14  diagnostics_summary                  median_duration_days   
15  diagnostics_summary                   n_negative_duration   
16  diagnostics_summary                    median_days_supply   
17  diagnostics_summary                       median_quantity   
18  diagnostics_summary                  proportion_with_dose   
19  diagnostics_summary  proportion_verbatim_end_date_missing   
20  diagnostics_summary                             n_records   
21  diagnostics_summary                              n_sample   
22  diagnostics_summary                             n_persons   
23  diagnostics_summary               mean_proportion_missing   
24  diagnostics_summary                  median_duration_days   
25  diagnostics_summary                   n_negative_duration   
26  diagnostics_summary                    median_days_supply   
27  diagnostics_summary                       median_quantity   
28  diagnostics_summary                  proportion_with_dose   
29  diagnostics_summary  proportion_verbatim_end_date_missing   

                           estimate_name  \
0                              n_records   
1                               n_sample   
2                              n_persons   
3                mean_proportion_missing   
4                   median_duration_days   
5                    n_negative_duration   
6                     median_days_supply   
7                        median_quantity   
8                   proportion_with_dose   
9   proportion_verbatim_end_date_missing   
10                             n_records   
11                              n_sample   
12                             n_persons   
13               mean_proportion_missing   
14                  median_duration_days   
15                   n_negative_duration   
16                    median_days_supply   
17                       median_quantity   
18                  proportion_with_dose   
19  proportion_verbatim_end_date_missing   
20                             n_records   
21                              n_sample   
22                             n_persons   
23               mean_proportion_missing   
24                  median_duration_days   
25                   n_negative_duration   
26                    median_days_supply   
27                       median_quantity   
28                  proportion_with_dose   
29  proportion_verbatim_end_date_missing   

   [header_name]cdm_name\n[header_level]mock_cdm ingredient_concept_id  \
0                                            500               1125315   
1                                            500               1125315   
2                                            250               1125315   
3                                           0.07               1125315   
4                                          55.40               1125315   
5                                              0               1125315   
6                                          20.50               1125315   
7             

## 10. Benchmarking

`benchmark_drug_diagnostics` runs `execute_checks` multiple times and reports timing statistics.
Useful for profiling with different configurations.

In [33]:
bench = benchmark_drug_diagnostics(
    cdm,
    ingredient_concept_ids=[1367500],
    checks=["missing", "exposure_duration", "type"],
    sample_size=None,
    n_runs=3,
)
print(bench)

shape: (3, 5)
┌─────┬───────────────────────┬───────────────────┬───────────┬────────────────────────┐
│ run ┆ ingredient_concept_id ┆ ingredient        ┆ n_records ┆ execution_time_seconds │
│ --- ┆ ---                   ┆ ---               ┆ ---       ┆ ---                    │
│ i64 ┆ i64                   ┆ str               ┆ i64       ┆ f64                    │
╞═════╪═══════════════════════╪═══════════════════╪═══════════╪════════════════════════╡
│ 1   ┆ 1367500               ┆ Unknown (1367500) ┆ 0         ┆ 0.028                  │
│ 2   ┆ 1367500               ┆ Unknown (1367500) ┆ 0         ┆ 0.025                  │
│ 3   ┆ 1367500               ┆ Unknown (1367500) ┆ 0         ┆ 0.027                  │
└─────┴───────────────────────┴───────────────────┴───────────┴────────────────────────┘


## 11. Summary

This notebook covered the full `omopy.drug_diagnostics` workflow:

1. **`AVAILABLE_CHECKS`** — 12 diagnostic checks covering missing values, exposure duration, drug types, routes, source concepts, days supply, verbatim end dates, dose coverage, sig, quantity, days between records, and an overall summary.
2. **`execute_checks`** — runs diagnostics for one or more ingredients, returning a `DiagnosticsResult` with dict-like access to per-check Polars DataFrames.
3. **`summarise_drug_diagnostics`** — converts results into the 13-column `SummarisedResult` format.
4. **`table_drug_diagnostics`** — formats results as polars DataFrames or `great_tables` GT objects.
5. **`plot_drug_diagnostics`** — bar charts for categorical checks, box plots for quantile checks.
6. **`mock_drug_exposure`** — generates realistic synthetic diagnostics for testing and demos.
7. **`benchmark_drug_diagnostics`** — profiles execution time across runs.

**Next steps:**
- Combine with `omopy.drug` for drug utilisation analysis
- Use `omopy.treatment` for treatment pathway analysis on the same ingredients
- Export `SummarisedResult` to CSV/parquet for downstream reporting